In [ ]:
import argparse
import torch
import os
import io
import sys
import re
import pandas as pd
from contextlib import redirect_stdout

from exp.exp_informer import Exp_Informer


CUDA available: True
GPU: Tesla T4


In [ ]:
args = argparse.Namespace()

# ===== Model & data =====
args.model = 'informer'
args.data = 'ETTh1'
args.root_path = './data/ETT/'
args.data_path = 'ETTh1.csv'
args.features = 'M'
args.target = 'OT'
args.freq = 'h'
args.checkpoints = './checkpoints/'

# ===== Sequence length =====
args.seq_len = 96
args.label_len = 48
args.pred_len = 24

# ===== Model size =====
args.enc_in = 7
args.dec_in = 7
args.c_out = 7
args.d_model = 512
args.n_heads = 8
args.e_layers = 2
args.d_layers = 1
args.s_layers = '3,2,1'
args.d_ff = 2048
args.factor = 5
args.padding = 0
args.distil = True
args.dropout = 0.05
args.attn = 'prob'
args.embed = 'timeF'
args.activation = 'gelu'
args.output_attention = False
args.mix = True

# ===== Training =====
args.num_workers = 0
args.itr = 1
args.train_epochs = 6
args.batch_size = 32
args.patience = 3
args.learning_rate = 0.0001
args.des = 'colab_run'
args.loss = 'mse'
args.lradj = 'type1'
args.use_amp = False
args.inverse = False

# ===== GPU =====
args.use_gpu = torch.cuda.is_available()
args.gpu = 0
args.use_multi_gpu = False
args.devices = '0'


In [ ]:
data_parser = {
    'ETTh1':{'data':'ETTh1.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTh2':{'data':'ETTh2.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTm1':{'data':'ETTm1.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTm2':{'data':'ETTm2.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'WTH':{'data':'WTH.csv','T':'WetBulbCelsius','M':[12,12,12],'S':[1,1,1],'MS':[12,12,1]},
    'ECL':{'data':'ECL.csv','T':'MT_320','M':[321,321,321],'S':[1,1,1],'MS':[321,321,1]},
    'Solar':{'data':'solar_AL.csv','T':'POWER_136','M':[137,137,137],'S':[1,1,1],'MS':[137,137,1]},
}

if args.data in data_parser:
    info = data_parser[args.data]
    args.data_path = info['data']
    args.target = info['T']
    args.enc_in, args.dec_in, args.c_out = info[args.features]

args.s_layers = [int(s) for s in args.s_layers.split(',')]
args.detail_freq = args.freq
args.freq = args.freq[-1:]

print("Args in experiment:")
print(args)


In [ ]:
pred_len_list = [24, 48, 168, 336, 720]

log_dir = "./logs"
os.makedirs(log_dir, exist_ok=True)

txt_log = os.path.join(log_dir, "pred_len_results.txt")
csv_log = os.path.join(log_dir, "pred_len_results.csv")

with open(txt_log, "w") as f:
    f.write("Informer ETTh1 results\n")
    f.write("pred_len | MSE | MAE\n")
    f.write("=" * 40 + "\n")

results = []
print("Logging to:", txt_log)


In [ ]:
Exp = Exp_Informer

for ii in range(args.itr):
    setting = '{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_at{}_fc{}_eb{}_dt{}_mx{}_{}_{}'.format(
        args.model, args.data, args.features,
        args.seq_len, args.label_len, args.pred_len,
        args.d_model, args.n_heads, args.e_layers,
        args.d_layers, args.d_ff, args.attn,
        args.factor, args.embed, args.distil,
        args.mix, args.des, ii
    )

    exp = Exp(args)

    print(f'>>>>>>> start training : {setting} >>>>>>>>>>>>>>>>>>>>>>')
    exp.train(setting)

    print(f'>>>>>>> testing : {setting} <<<<<<<<<<<<<<<<<<<<<<<<<<<<')
    exp.test(setting)

    torch.cuda.empty_cache()
